In [ ]:
import os
from dotenv import load_dotenv
from datetime import datetime

load_dotenv(override=True)

In [ ]:
open_router_api_key = os.getenv("OPEN_ROUTER_API_KEY")

if open_router_api_key is None:
    raise ValueError("OPEN_ROUTER_API_KEY environment variable is not set")
else:
    print("OPEN_ROUTER_API_KEY environment variable is set")

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name="openrouter/free",
    api_key=open_router_api_key,
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
date = datetime.now().strftime("%Y-%m-%d")
time = datetime.now().strftime("%H:%M:%S")

SYSTEM_PROMPT = f"""you are a cricket assistant that can answer questions about cricket. if you don't know the answer,
please say you don't know. if you are unsure of the answer, 
please say you are unsure. you can only answer questions about cricket.

Today's data is: {date}
Today's time is: {time}
"""
USER_PROMPT = "How many india won the one day world cup trophy?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT},
]

In [ ]:
print(SYSTEM_PROMPT)

In [ ]:
def call_llm(messages):
    response = llm.invoke(messages)
    is_safe = output_guard(response.content)
    if not is_safe.is_safe:
        return "The output was blocked by the guardrail"
    return response.content


In [ ]:
SYSTEM_PROMPT_FOR_INPUT_GUARDRAILS = """
    You are a input guardrail.
    If user input is not related to the cricket, you can return a response as False or else you can return a response as True.
"""

In [ ]:
from pydantic import BaseModel, Field

class GuardrailForInput(BaseModel):
    is_safe: bool = Field(description="Whether the input is safe or not")

In [ ]:
class GuardrailForOutput(BaseModel):
    is_safe: bool = Field(description="Whether the output is safe or not")

In [ ]:
guardrails_llm = ChatOpenAI(
    model_name="openrouter/free",
    api_key=open_router_api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.5,
)

structured_input_guardrails_llm = guardrails_llm.with_structured_output(GuardrailForInput)
structured_output_guardrails_llm = guardrails_llm.with_structured_output(GuardrailForOutput)

In [ ]:
# input guardrails

def input_guardrails(input):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_FOR_INPUT_GUARDRAILS},
        {"role": "user", "content": input},
    ]
    response = structured_input_guardrails_llm.invoke(messages)
    return response

In [ ]:
def output_guard(response: str):
    messages = [
        {"role": "system", "content": "Your are an output guardrails. You will check if the output is safe."},
        {"role": "user", "content": response},
    ]
    resp = structured_output_guardrails_llm.invoke(messages)
    print(resp)
    return resp;
    

In [ ]:
input = "tell me about Rohit Sharma"
response = input_guardrails(input).is_safe
print(response)
if response:
    print("Safe")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": input}, 
    ]
    print(call_llm(messages))
else:
    print("Invalid input")